# NB03 - Train-Only Normalization + PyTorch Dataset/QC (no training)

**Responsibilities:** (1) normalization statistics from TRAIN ONLY, (2) reusable `FloodDataset`/DataLoaders, (3) rigorous pre-training QC. **No model training, no augmentation/cropping/balancing, no .npz modification.**

Anti-leakage: stats loop touches `processed/train/` only; valid/test/bolivia are normalized with the SAME saved train statistics. Normalization is applied dynamically at load time - NB02 files stay physical/raw.

## 1. Load project context

In [ ]:
from pathlib import Path
CTX = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao/project_context")
for f in ["PROJECT_STATE.md", "DECISIONS.md", "NEXT_STEPS.md"]:
    print("=" * 70)
    print(f"### {f} (head) ###")
    print("\n".join((CTX / f).read_text(encoding="utf-8").splitlines()[:14]))
print("\nDATASET_SPEC (NB02 processed spec):")
print("\n".join([ln for ln in (CTX / "DATASET_SPEC.md").read_text(encoding="utf-8").splitlines() if "processed/" in ln or "CONFIRMED" in ln][:6]))


## 2. Configuration
Single place for every knob. Batch size is conservative for RTX 4050 (6 GB VRAM) / 16 GB RAM: one sample X is 6x512x512 f32 (~6.3 MB), so batch 4 is ~25 MB + overhead.

In [ ]:
from pathlib import Path
import numpy as np, torch
PROJECT_ROOT = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao")
PROCESSED_ROOT = PROJECT_ROOT / "processed"
NORM_STATS_PATH = PROCESSED_ROOT / "normalization_stats.json"
QC_SUMMARY_PATH = PROCESSED_ROOT / "dataset_qc_summary.json"
CHANNEL_NAMES = ["VV_dB", "VH_dB", "DEM_m", "rain_t-3", "rain_t-2", "rain_t-1"]
EXPECTED = {"train": 252, "valid": 89, "test": 90, "bolivia": 15}
BATCH_SIZE = 4
NUM_WORKERS = 0      # Windows reliability: no multiprocessing complexity
PIN_MEMORY = torch.cuda.is_available()
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
g = torch.Generator().manual_seed(SEED)  # deterministic train shuffling
print(f"torch {torch.__version__} | cuda={torch.cuda.is_available()} | batch={BATCH_SIZE} workers={NUM_WORKERS} pin={PIN_MEMORY} seed={SEED}")


## 3. Load processed dataset metadata
File lists come from `processed/<split>/*.npz` cross-checked against `metadata.csv`.

In [ ]:
import pandas as pd
meta = pd.read_csv(PROCESSED_ROOT / "metadata.csv")
files = {}
for split in EXPECTED:
    files[split] = sorted((PROCESSED_ROOT / split).glob("*.npz"))
    print(f"{split}: {len(files[split])} npz files (expected {EXPECTED[split]}) | metadata.csv rows: {(meta['split'] == split).sum()}")
    assert len(files[split]) == EXPECTED[split]
    assert (meta["split"] == split).sum() == EXPECTED[split]
    assert {p.stem for p in files[split]} == set(meta.loc[meta['split'] == split, 'chip_id']), f"csv/dir mismatch in {split}"
print("File inventory matches metadata.csv for all splits.")


## 4. Dataset integrity checks
One sample at a time (never the whole dataset in RAM): shapes, dtypes, finiteness, y/mask contract.

In [ ]:
import numpy as np
rows = []
for split, paths in files.items():
    bad = []
    for p in paths:
        d = np.load(p, allow_pickle=True)
        X, y, vm = d["X"], d["y"], d["valid_mask"]
        ok = (X.shape == (6, 512, 512) and X.dtype == np.float32 and y.shape == (512, 512)
              and y.dtype == np.uint8 and vm.shape == (512, 512) and vm.dtype == bool
              and bool(np.isfinite(X).all())
              and set(np.unique(y[vm.astype(bool)]).tolist()) <= {0, 1}
              and bool((y[~vm.astype(bool)] == 0).all()))
        if not ok:
            bad.append(p.name)
    rows.append({"split": split, "checked": len(paths), "bad": len(bad), "examples": bad[:3]})
    print(f"{split}: {len(paths)} checked, bad={len(bad)}")
    assert not bad, bad[:3]
print("INTEGRITY: all 446 samples pass (shapes/dtypes/finite X, y in {0,1} on valid, y==0 where masked).")


## 5. Train-only normalization statistics
**Leakage-critical cell:** iterates `processed/train/` ONLY. Statistics are accumulated in float64 with sum/sum-of-squares/count per channel over `valid_mask` pixels (this excludes label-NoData AND every NB02-filled value, since fills are masked invalid) - numerically stable, one chip at a time, no giant concatenation.

In [ ]:
import numpy as np, datetime
train_paths = files["train"]  # ONLY train - valid/test/bolivia never touched here
assert all("processed" in str(p) and "/train/" in str(p).replace("\\", "/") for p in train_paths), "train-dir confinement broken"
n_ch = len(CHANNEL_NAMES)
cnt = np.zeros(n_ch, dtype=np.float64)
s1 = np.zeros(n_ch, dtype=np.float64)   # sum
s2 = np.zeros(n_ch, dtype=np.float64)   # sum of squares
cmin = np.full(n_ch, np.inf); cmax = np.full(n_ch, -np.inf)
for p in train_paths:
    d = np.load(p, allow_pickle=True)
    X = d["X"].astype(np.float64)
    vm = d["valid_mask"].astype(bool)
    for c in range(n_ch):
        v = X[c][vm]
        v = v[np.isfinite(v)]  # belt-and-braces; NB02 X is finite, mask excludes fills
        if v.size == 0:
            continue
        cnt[c] += v.size; s1[c] += v.sum(); s2[c] += (v ** 2).sum()
        cmin[c] = min(cmin[c], float(v.min())); cmax[c] = max(cmax[c], float(v.max()))
assert (cnt > 0).all(), "a channel has zero valid train pixels"
mean = s1 / cnt
var = np.maximum(s2 / cnt - mean ** 2, 0.0)
std = np.sqrt(var)
assert (std > 0).all(), f"degenerate std: {std}"
stats_df = pd.DataFrame({"channel": CHANNEL_NAMES, "mean": mean.round(6), "std": std.round(6),
                         "min": np.round(cmin, 4), "max": np.round(cmax, 4), "valid_count": cnt.astype(np.int64)})
display(stats_df)
print(f"Accumulated over {len(train_paths)} TRAIN chips only; total valid px per channel ~ {cnt[0]:,.0f}")


## 6. Save normalization statistics
Written once; reload-verified. NB02 `.npz` files are never altered.

In [ ]:
import json, datetime
norm_stats = {
    "split": "train", "n_chips": len(train_paths),
    "channels": CHANNEL_NAMES,
    "mean": [float(x) for x in mean], "std": [float(x) for x in std],
    "min": [float(x) for x in cmin], "max": [float(x) for x in cmax],
    "valid_count": [int(x) for x in cnt],
    "method": "standardization x_norm=(x-mean)/std, float64 sum/sumsq accumulation over valid_mask pixels, one chip at a time",
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "torch_version": torch.__version__, "seed": SEED,
}
NORM_STATS_PATH.write_text(json.dumps(norm_stats, indent=2))
re = json.loads(NORM_STATS_PATH.read_text())
assert re["split"] == "train" and re["n_chips"] == 252 and len(re["mean"]) == 6
print(f"Saved + reload-verified: {NORM_STATS_PATH}")
MEAN = np.array(re["mean"], dtype=np.float32).reshape(-1, 1, 1)
STD = np.array(re["std"], dtype=np.float32).reshape(-1, 1, 1)


## 7. FloodDataset implementation
Generic, reusable by NB04/NB05. Ordering contract (documented): load physical X → standardize with TRAIN stats → sanitize any residual non-finite to `0.0` (= channel mean in normalized space, a neutral value) → return finite tensors. `valid_mask` (from NB02) is passed through untouched and must gate the loss. `y` is float32 (BCE/Dice-ready: 0.0/1.0); masked positions read 0.0. No augmentation/cropping/balancing here.

In [ ]:
import numpy as np, torch
from torch.utils.data import Dataset

class FloodDataset(Dataset):
    """Multimodal flood chips. Split in {train,valid,test,bolivia}; normalization ALWAYS uses saved TRAIN stats.
    Returns dict(x: f32 (6,512,512) normalized finite, y: f32 (512,512), mask: bool (512,512), chip_id: str)."""
    def __init__(self, split, processed_root=PROCESSED_ROOT, stats_path=NORM_STATS_PATH):
        assert split in ("train", "valid", "test", "bolivia"), split
        st = __import__("json").loads(stats_path.read_text())
        assert st["split"] == "train", "normalization stats must be train-derived"
        self.mean = np.array(st["mean"], dtype=np.float32).reshape(-1, 1, 1)
        self.std = np.array(st["std"], dtype=np.float32).reshape(-1, 1, 1)
        assert (self.std > 0).all()
        self.paths = sorted((processed_root / split).glob("*.npz"))
        assert len(self.paths) > 0, f"no samples for split {split}"
        self.split = split

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        d = np.load(self.paths[i], allow_pickle=True)
        x = (d["X"].astype(np.float32) - self.mean) / self.std           # standardize physical values
        x = np.where(np.isfinite(x), x, 0.0).astype(np.float32)            # sanitize: non-finite -> 0 (=mean)
        y = d["y"].astype(np.float32)                                    # 0.0/1.0, 0 where masked (NB02)
        m = d["valid_mask"].astype(bool)                                # loss gate, untouched
        return {"x": torch.from_numpy(x), "y": torch.from_numpy(y),
                "mask": torch.from_numpy(m), "chip_id": str(d["chip_id"])}

ds = {s: FloodDataset(s) for s in ("train", "valid", "test", "bolivia")}
print({s: len(ds[s]) for s in ds})
assert {s: len(ds[s]) for s in ds} == EXPECTED
r = ds["train"][0]
print({k: (tuple(v.shape), str(v.dtype)) for k, v in r.items() if k != "chip_id"}, "chip:", r["chip_id"])


## 8. DataLoader implementation
Conservative: batch 4, `num_workers=0` (Windows-safe), pin_memory iff CUDA, train shuffled with seeded generator.

In [ ]:
from torch.utils.data import DataLoader
loaders = {
    "train": DataLoader(ds["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY, generator=g),
    "valid": DataLoader(ds["valid"], batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
    "test": DataLoader(ds["test"], batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
    "bolivia": DataLoader(ds["bolivia"], batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
}
b = next(iter(loaders["train"]))
print({k: (tuple(v.shape) if hasattr(v, "shape") else v) for k, v in b.items() if k != "chip_id"}, "| chips:", b["chip_id"][:2])
assert tuple(b["x"].shape) == (BATCH_SIZE, 6, 512, 512) and tuple(b["y"].shape) == (BATCH_SIZE, 512, 512)
assert tuple(b["mask"].shape) == (BATCH_SIZE, 512, 512) and b["mask"].dtype == torch.bool
print("Batch contract OK.")


## 9. Batch-level QC
Lengths, shapes, finiteness, mask dtype, y-contract on every batch of every split (streamed, low RAM).

In [ ]:
import torch
qc_rows = []
for split, dl in loaders.items():
    n_b, n_s, bad = 0, 0, []
    for b in dl:
        n_b += 1; n_s += b["x"].shape[0]
        m = b["mask"]
        ok = (tuple(b["x"].shape[1:]) == (6, 512, 512) and b["x"].dtype == torch.float32
              and bool(torch.isfinite(b["x"]).all()) and m.dtype == torch.bool
              and set(torch.unique(b["y"][m]).tolist()) <= {0.0, 1.0}
              and bool(((b["y"][~m]) == 0).all()))
        if not ok:
            bad.append(n_b)
    qc_rows.append({"split": split, "len_dataset": len(ds[split]), "samples_seen": n_s, "batches": n_b, "bad_batches": len(bad)})
    print(f"{split}: samples={n_s} batches={n_b} bad={len(bad)}")
    assert n_s == EXPECTED[split] and not bad
print("BATCH QC: all splits pass (finite X, bool mask, y in {0,1} on valid, 0 where masked).")


## 10. Normalization QC
Train channels (valid pixels) must sit at mean≈0/std≈1 (float64 accumulation over streamed batches; tolerance 5e-2 for the mean, 5% relative for std - heavy-tailed rain/DEM legitimately deviate from unit Gaussianity, but standardization centers them). Other splits reuse the SAME train stats (spot-checked finite).

In [ ]:
import numpy as np
cnt = np.zeros(6); s1 = np.zeros(6); s2 = np.zeros(6)
for b in loaders["train"]:
    x = b["x"].numpy().astype(np.float64); m = b["mask"].numpy()
    for c in range(6):
        v = x[:, c][m]
        cnt[c] += v.size; s1[c] += v.sum(); s2[c] += (v ** 2).sum()
mn, sd = s1 / cnt, np.sqrt(np.maximum(s2 / cnt - (s1 / cnt) ** 2, 0))
ndf = pd.DataFrame({"channel": CHANNEL_NAMES, "norm_mean": np.round(mn, 4), "norm_std": np.round(sd, 4)})
display(ndf)
assert np.all(np.abs(mn) < 5e-2), mn
assert np.all(np.abs(sd - 1) < 5e-2), sd
for s in ("valid", "test", "bolivia"):
    b = next(iter(loaders[s]))
    assert bool(torch.isfinite(b["x"]).all()), s
    print(f"{s}: normalized with TRAIN stats, finite OK (spot mean={float(b['x'].mean()):.3f} - not expected to be 0)")
print("NORMALIZATION QC PASS.")


## 11. Visual QC
One representative chip per split (normalized VV/VH/DEM/rain-t-1 + y + mask), plus a raw-vs-normalized comparison proving spatial structure is preserved.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
lcmap = ListedColormap(["#1a1a1a", "#00b3ff"])
for s in ("train", "valid", "test", "bolivia"):
    r = ds[s][0]
    x, y, m = r["x"].numpy(), r["y"].numpy(), r["mask"].numpy()
    fig, ax = plt.subplots(1, 6, figsize=(16, 3))
    ax[0].imshow(x[0], cmap="gray"); ax[0].set_title(f"VVn m={x[0].mean():.2f}")
    ax[1].imshow(x[1], cmap="gray"); ax[1].set_title(f"VHn m={x[1].mean():.2f}")
    ax[2].imshow(x[2], cmap="terrain"); ax[2].set_title(f"DEMn m={x[2].mean():.2f}")
    ax[3].imshow(x[5], cmap="Blues"); ax[3].set_title(f"rain t-1n max={x[5].max():.2f}")
    ax[4].imshow(np.where(m, y, -1), cmap=ListedColormap(["#808080", "#1a1a1a", "#00b3ff"]), vmin=-1, vmax=1, interpolation="nearest")
    ax[4].set_title(f"y flood={(y[m]==1).mean()*100:.1f}% (grey=mask)")
    ax[5].imshow(m, cmap="gray"); ax[5].set_title(f"mask valid={m.mean()*100:.1f}%")
    for a in ax: a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{s}: {r['chip_id']} (train-stats normalized)")
    fig.tight_layout(); plt.show()
# raw vs normalized: spatial structure preserved?
raw = np.load(files["train"][0], allow_pickle=True)["X"]
nrm = ds["train"][0]["x"].numpy()
fig, ax = plt.subplots(2, 2, figsize=(9, 8))
ax[0, 0].imshow(raw[0], cmap="gray"); ax[0, 0].set_title(f"raw VV dB [{raw[0].min():.1f},{raw[0].max():.1f}]")
ax[0, 1].imshow(nrm[0], cmap="gray"); ax[0, 1].set_title(f"normalized VV [{nrm[0].min():.2f},{nrm[0].max():.2f}]")
ax[1, 0].imshow(raw[2], cmap="terrain"); ax[1, 0].set_title(f"raw DEM m [{raw[2].min():.1f},{raw[2].max():.1f}]")
ax[1, 1].imshow(nrm[2], cmap="terrain"); ax[1, 1].set_title(f"normalized DEM [{nrm[2].min():.2f},{nrm[2].max():.2f}]")
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
fig.suptitle("Raw (physical) vs normalized - structure preserved, scale standardized")
fig.tight_layout(); plt.show()
print("Visual QC done - inspect panels above before NB04.")


## 12. Leakage / reproducibility checks
Hard gates: stats claim train-only AND the stats loop provably enumerated train files only.

In [ ]:
import json
st = json.loads(NORM_STATS_PATH.read_text())
checks = []
def check(name, cond, detail=""):
    checks.append({"check": name, "status": "PASS" if bool(cond) else "FAIL", "detail": detail})
check("stats['split']=='train'", st["split"] == "train", st["split"])
check("stats n_chips==252", st["n_chips"] == 252, str(st["n_chips"]))
check("stats cover only train files", len(train_paths) == 252 and all("/train/" in str(p).replace("\\", "/") for p in train_paths), f"{len(train_paths)} files")
check("no per-split stats", set(st.keys()) >= {"mean", "std"} and not any(k in st for k in ("valid_stats", "test_stats", "bolivia_stats", "per_split")), sorted(st.keys()))
check("all splits share identical norm vectors", all(np.allclose(ds[s].mean, MEAN) and np.allclose(ds[s].std, STD) for s in ds), "FloodDataset asserts train-derived stats")
check("raw npz unmodified (spot mtime/size sane)", all(p.stat().st_size > 1_000_000 for p in files["train"][:5]), "compressed npz intact")
check("seeded config recorded", st["seed"] == SEED, f"seed={st['seed']}")
cdf = pd.DataFrame(checks); display(cdf)
assert (cdf.status == "PASS").all(), "LEAKAGE GATE FAILURE"
print("LEAKAGE/REPRO: ALL PASS - normalization is train-only (252 chips), all splits share it.")


## 13. Final summary
`dataset_qc_summary.json` + console report. No training happened here.

In [ ]:
import json, datetime
summary = {
    "notebook": "NB03_Dataset_QC_and_Dataloaders", "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "counts": {s: len(ds[s]) for s in ds}, "normalization": {"strategy": "(x-mean)/std", "stats_file": str(NORM_STATS_PATH),
    "stats_split": st["split"], "stats_n_chips": st["n_chips"], "channels": CHANNEL_NAMES,
    "channel_mean": st["mean"], "channel_std": st["std"]},
    "dataloader": {"batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS, "pin_memory": PIN_MEMORY,
                     "train_shuffle": True, "eval_shuffle": False, "seed": SEED},
    "tensors": {"x": "float32 (B,6,512,512) normalized finite", "y": "float32 (B,512,512) 0/1 (0 where masked)", "mask": "bool (B,512,512)"},
    "qc": "integrity + batch + normalization + visual + leakage gates ALL PASS",
}
QC_SUMMARY_PATH.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("\nNB03 COMPLETE: normalization (train-only) + FloodDataset/DataLoaders + QC. Ready for NB04 (modeling).")


## 14. Project context update

In [ ]:
import datetime, re
now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
entry = (f"\n## {now} - NB03_Dataset_QC_and_Dataloaders COMPLETE\n"
         f"- Train-only stats over 252 train chips (valid_mask px, float64 accum): " + ", ".join(f"{c} mu={m:.3f} sd={s:.3f}" for c, m, s in zip(CHANNEL_NAMES, st['mean'], st['std'])) + "\n"
         f"- Saved processed/normalization_stats.json (split=train, verified) + dataset_qc_summary.json\n"
         f"- FloodDataset (dict x/y/mask/chip_id; y float32; sanitize non-finite->0 post-norm) + loaders (batch={BATCH_SIZE}, workers={NUM_WORKERS}, pin={PIN_MEMORY}, train shuffle seeded)\n"
         f"- QC: integrity 446/446, batch gates all splits, norm mean~0/std~1 on train, visual raw-vs-norm OK, leakage gates ALL PASS\n"
         f"- No training/augmentation/cropping/balancing; .npz untouched. NB03 safe to lock; next NB04 modeling.\n")
(CTX / "CHANGELOG.md").write_text((CTX / "CHANGELOG.md").read_text(encoding="utf-8") + entry, encoding="utf-8")
ps = CTX / "PROJECT_STATE.md"
t = ps.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now} (NB03 complete: yes)", t, count=1, flags=re.M)
t += f"\n\n## NB03 result ({now})\n- Train-only norm stats (252 chips) + FloodDataset/loaders; all QC + leakage gates PASS.\n"
ps.write_text(t, encoding="utf-8")
dc = CTX / "DECISIONS.md"
t = dc.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now} (D11 normalization)", t, count=1, flags=re.M)
t += "\n## D11 - Standardization with train-only stats, dynamic at load (" + now + ")\nWhy: (x-mean)/std per channel from 252 train chips (valid_mask px); y/mask never normalized; .npz stay physical; post-norm sanitize non-finite->0 (=mean); y float32 for BCE/Dice; batch 4 / workers 0 / pin iff CUDA.\n"
dc.write_text(t, encoding="utf-8")
nx = CTX / "NEXT_STEPS.md"
t = nx.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now}", t, count=1, flags=re.M)
t += f"\n\n## Update ({now})\n- NB03 COMPLETE + validated. Next: NB04 model + training (loss must use valid_mask; handle ~10% flood imbalance).\n"
nx.write_text(t, encoding="utf-8")
print(entry)
print("Context files updated:", now)
